## Inspecting the Result

Read Gold back out of SQLite, one DataFrame per table. `read_table` is the only reader; the cells below just call it.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_ROOT = WORKDIR / "medallion_extraction"
LOCAL_DB_PATH = PROJECT_ROOT / "extraction_local.sqlite"
INSPECT = sqlite3.connect(f"{LOCAL_DB_PATH.as_uri()}?mode=ro", uri=True)

GOLD_TABLES = [
    "papers",
    "sections",
    "tables",
    "figures",
    "experiments",
    "ingredients",
    "experiment_ingredients",
    "indicators",
    "measurements",
    "evidence",
]


def read_table(name: str, limit: int | None = None) -> pd.DataFrame:
    """One table as a DataFrame. The name is checked against the schema rather than
    interpolated blind: SQLite cannot parameterise an identifier, so an allowlist is what
    keeps this from being a string-built query over arbitrary input."""
    if name not in GOLD_TABLES:
        raise ValueError(f"{name!r} is not a Gold table; expected one of {GOLD_TABLES}")
    query = f"SELECT * FROM {name}" + (f" LIMIT {int(limit)}" if limit else "")
    frame = pd.read_sql_query(query, INSPECT)
    print(f"{name}: {len(frame)} rows x {frame.shape[1]} columns")
    return frame

In [ ]:
papers = read_table("papers")
papers

papers: 3 rows x 11 columns


,id,doi,title,abstract,published_year,source_path,file_hash,bronze_path,silver_path,markdown_path,created_at
0,1,None,11,None,None,11.pdf,1ef42b74924e18a8c165020bdc174bc36eef4df531e32c...,/content/medallion_extraction/artifacts/bronze/11,/content/medallion_extraction/artifacts/silver/11,/content/medallion_extraction/artifacts/bronze...,2026-07-29 15:49:15.041834
1,2,None,12,None,None,12.pdf,ab392d00aa886dcd514e0e0322a06a042e164dd0442b59...,/content/medallion_extraction/artifacts/bronze/12,/content/medallion_extraction/artifacts/silver/12,/content/medallion_extraction/artifacts/bronze...,2026-07-29 17:11:50.513255
2,3,None,13,None,None,13.pdf,34ef90967c8ed0720ad94cfbee6f87db8f4764b4bcfe58...,/content/medallion_extraction/artifacts/bronze/13,/content/medallion_extraction/artifacts/silver/13,/content/medallion_extraction/artifacts/bronze...,2026-07-29 17:15:23.306149


In [ ]:
sections = read_table("sections")
sections

sections: 104 rows x 7 columns


,id,paper_id,section_title,content_markdown,embedding,section_order,docling_item_ref
0,1,1,Document,<!-- image -->\n\nhttp://pubs.acs.org/journal/...,None,1,#/sections/0
1,2,1,Origanum majorana L. Essential Oil-Coated Pape...,"Sulhattin Yasar, Nizam Mustafa Nizaml ı og ̆ l...",None,2,#/sections/1
2,3,1,ACCESS,[Metrics &amp; More](https://pubs.acs.org/doi/...,None,3,#/sections/2
3,4,1,■ INTRODUCTION,During refrigeration of processed meat product...,None,4,#/sections/3
4,5,1,■ MATERIALS AND METHODS,Materials. OmEO was purchased from a commercia...,None,5,#/sections/4
...,...,...,...,...,...,...,...
99,100,3,3.2.2. Antimicrobial Effect of MLE against S. ...,There was no significant difference ( p &gt; 0...,None,15,#/sections/14
100,101,3,3.2.3. Antimicrobial Effect of MLE against S. ...,The current study revealed no significant diff...,None,16,#/sections/15
101,102,3,3.3. Sensory Evaluation,Moringa oleifera is a nutraceutical element ri...,None,17,#/sections/16
102,103,3,References,"1. Behbahani, B.A.; Noshad, M.; Jooyandeh, H. ...",None,18,#/sections/17


In [ ]:
tables = read_table("tables")
tables

tables: 7 rows x 7 columns


,id,paper_id,caption,csv_filepath,structured_json,docling_item_ref,created_at
0,1,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 15:49:15.046332
1,2,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 16:34:45.595955
2,3,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/1,2026-07-29 16:50:27.261946
3,4,1,"Table 2. Changes in Hardness, pH, HCl Titrate,...",/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""keyed: the ...",#/tables/1,2026-07-29 16:58:07.661677
4,5,2,Table 1 Chemical composition of TEO and SEO me...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": false, ""reason"": ""no column ...",#/tables/0,2026-07-29 17:11:50.911097
5,6,3,Table 1. Effect of Moringa oleifera leaves ext...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""long: one a...",#/tables/0,2026-07-29 17:15:23.310279
6,7,3,Table 2. * Mean values of the sensory characte...,/content/medallion_extraction/artifacts/silver...,"{""gate"": {""fits"": true, ""reason"": ""keyed: the ...",#/tables/1,2026-07-29 17:15:23.310284


In [ ]:
figures = read_table("figures")
figures

figures: 14 rows x 7 columns


,id,paper_id,caption,image_filepath,semantic_tags_json,docling_item_ref,created_at
0,1,1,"Figure 3. DPPH scavenging activity (%), degree...",/content/medallion_extraction/artifacts/bronze...,[],#/pictures/11,2026-07-29 16:58:07.528143
1,2,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/3,2026-07-29 17:11:50.814418
2,3,2,Fig. 2. Changes in TVB-N values of chicken fil...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/4,2026-07-29 17:11:50.814425
3,4,2,Fig. 3. Changes in PV values of chicken ...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/5,2026-07-29 17:11:50.814429
4,5,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/6,2026-07-29 17:11:50.814432
5,6,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/7,2026-07-29 17:11:50.814435
6,7,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/8,2026-07-29 17:11:50.814438
7,8,2,Fig. 6. Changes in percentage of cooking loss ...,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/9,2026-07-29 17:11:50.814441
8,9,2,Fig. 7. Sensory evaluation of chicken fillets....,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/10,2026-07-29 17:11:50.814444
9,10,2,None,/content/medallion_extraction/artifacts/bronze...,[],#/pictures/11,2026-07-29 17:11:50.814447


In [ ]:
experiments = read_table("experiments")
experiments

experiments: 16 rows x 4 columns


,experiment_id,meat_matrix,treatment,weight_g
0,1,Ground beef,Minced beef was packed in parchment paper coat...,NaN
1,2,Ground beef,Minced beef was packed in parchment paper coat...,NaN
2,3,Ground beef,Minced beef was packed in parchment paper coat...,NaN
3,4,Chicken fillet,None,NaN
4,5,Chicken fillet,None,NaN
5,6,Chicken fillet,None,NaN
6,7,Chicken fillet,None,NaN
7,8,Chicken fillet,None,NaN
8,9,Chicken fillet,None,NaN
9,10,Chicken fillet,None,NaN


In [ ]:
ingredients = read_table("ingredients")
ingredients

ingredients: 31 rows x 4 columns


,ingredient_id,ingredient_name,functional_class,source
0,1,Thyme essential oil,essential oil,plant
1,2,Sage essential oil,essential oil,plant
2,3,α-Thujene,essential oil,plant
3,4,α-Pinene,essential oil,plant
4,5,Camphene,essential oil,plant
5,6,β-Pinene,essential oil,plant
6,7,3-Octanone,essential oil,plant
7,8,Myrcene,essential oil,plant
8,9,α-Terpinene,essential oil,plant
9,10,p-Cymene,essential oil,plant


In [ ]:
experiment_ingredients = read_table("experiment_ingredients")
experiment_ingredients

experiment_ingredients: 39 rows x 4 columns


,experiment_id,ingredient_id,concentration,concentration_unit
0,5,1,2.00,%
1,6,1,1.00,%
2,7,2,2.00,%
3,8,2,1.00,%
4,9,1,2.00,%
5,9,2,2.00,%
6,10,1,1.00,%
7,10,2,1.00,%
8,11,3,0.20,%
9,11,4,2.21,%


In [ ]:
indicators = read_table("indicators")
indicators

indicators: 21 rows x 5 columns


,indicator_id,indicator_name,indicator_type,indicator_unit,indicator_threshold
0,1,pH,chemical,pH,NaN
1,2,HCl titrate value,chemical,mL/g,NaN
2,3,DM,chemical,%,NaN
3,4,Water activity,chemical,aw,NaN
4,5,Total viable count,microbial,log CFU/g,7.0
5,6,"DPPH scavenging activity, %",chemical,unspecified,NaN
6,7,"TBA, gram MAD/kg dry matter",chemical,unspecified,NaN
7,8,"Peroxide value, mEq/kg sample",chemical,unspecified,NaN
8,9,unresolved indicator,chemical,unspecified,NaN
9,10,Total volatile basic nitrogen,chemical,mg N/100 g,25.0


In [ ]:
measurements = read_table("measurements")
measurements

measurements: 366 rows x 4 columns


,experiment_id,day,indicator_id,indicator_value
0,1,0,1,5.64
1,1,0,2,0.10
2,1,0,3,34.39
3,1,0,4,0.96
4,1,0,5,2.47
...,...,...,...,...
361,16,6,18,6.61
362,16,6,19,7.18
363,16,6,21,7.09
364,16,6,20,7.24


In [ ]:
evidence = read_table("evidence")
evidence

evidence: 84 rows x 14 columns


,id,paper_id,entity_type,entity_key,field_name,page_number,source_type,source_label,exact_text,method,rationale,confidence,value_is_approximate,docling_item_ref
0,1,1,experiment,"{""experiment_id"": 1}",indicator_value,6.0,table,None,None,stated,None,0.5,0,#/tables/1
1,2,1,experiment,"{""experiment_id"": 1}",treatment,NaN,prose,None,"The coating process involved two steps. First,...",stated,The text describes application of the starch s...,0.5,0,#/sections/4
2,3,1,experiment,"{""experiment_id"": 1}",weight_g,NaN,prose,None,,inferred,No mass of a single minced beef unit is report...,0.3,0,#/sections/4
3,4,1,experiment,"{""experiment_id"": 2}",indicator_value,6.0,table,None,None,stated,None,0.5,0,#/tables/1
4,5,1,experiment,"{""experiment_id"": 2}",treatment,NaN,prose,None,"The temperature was then lowered to 70 °C, dur...",stated,The text specifies that OmEO was incorporated ...,0.5,0,#/sections/4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,80,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
80,81,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
81,82,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1
82,83,3,experiment,"{""experiment_id"": 16}",indicator_value,10.0,table,None,None,stated,None,0.5,0,#/tables/1


In [ ]:
INSPECT.close()
print("Inspection connection closed.")

Inspection connection closed.
